# soamp — embed organisms with DNABERT-S (one-off)

Builds `data/organism_16s_dnabert_s.json`: a frozen **DNABERT-S** embedding of
each organism's **16S rRNA gene** (768-d). DNABERT-S is species-aware (a
DNABERT-2 model trained contrastively so same-species sequences embed close),
the property an organism representation needs. The `dnabert_s_16s` organism
featurizer reads this small table at training time, so training never loads
the model.

**Why Colab and a separate notebook:** the checkpoint ships remote modeling code
that does not load under the repo's `transformers==5.x` (hard `triton` import,
`config.pad_token_id`, meta-device init). A Colab Linux GPU runtime has `triton`;
this notebook installs `transformers<5` + `einops` *in this runtime only*. The
checkpoint revision is pinned in `config/features/base.yaml`
(`organism_16s_embedding.revision`) because that code is executed with
`trust_remote_code=True`.

Run it once, download the JSON, **commit it** (`data/organism_16s_dnabert_s.json`), then
`pipeline/features/02_build_organism_vocab.py --config config/features/organism_dnabert.yaml`
bakes `data/organism_vocab_dnabert_s_16s.json` locally. Re-run only when
`config/organism_genomes/organism_genome_accessions.csv` gains organisms.

Needs Colab Secret `GITHUB_TOKEN` (read-only PAT for the private repo).

In [ ]:
import os, subprocess
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
REPO, BRANCH, WORKDIR = "LukaJinc/soamp", "main", "/content/soamp"
token = os.environ["GITHUB_TOKEN"]
authed_url, plain_url = f"https://{token}@github.com/{REPO}.git", f"https://github.com/{REPO}.git"

def run_git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, "***"), end="")
    return result

if not os.path.isdir(WORKDIR):
    if run_git("clone", "--depth", "1", "--branch", BRANCH, authed_url, WORKDIR).returncode != 0:
        raise RuntimeError("git clone failed -- check GITHUB_TOKEN")
else:  # reused runtime: never run a stale checkout
    run_git("-C", WORKDIR, "fetch", "--depth", "1", authed_url, BRANCH)
    run_git("-C", WORKDIR, "reset", "--hard", "FETCH_HEAD")
subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", plain_url], check=True)
os.chdir(WORKDIR)
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Install order matters: the repo's requirements first, then pin transformers
# DOWN (<5) for DNABERT-S's remote code. This runtime only.
!pip install -q -r scripts/colab/requirements-colab.txt
!pip install -q -e . --no-deps
!pip install -q "transformers<5" einops
import transformers, torch
print("transformers", transformers.__version__, "| cuda:", torch.cuda.is_available())
assert int(transformers.__version__.split(".")[0]) < 5

In [ ]:
# Fetches the 3 genomes (~13 MB) + their GFF annotations, extracts the 16S gene,
# embeds it. Runs in a fresh process, so the editable install is visible.
!python pipeline/features/04_embed_organism_16s_dnabert_s.py

In [ ]:
import json
import numpy as np

table = json.load(open("data/organism_16s_dnabert_s.json"))
print("checkpoint:", table["checkpoint"], "@", table["revision"][:10], "| dim:", table["embedding_dim"])
keys = sorted(table["entries"])
for k in keys:
    e = table["entries"][k]
    print(f"  {k:30s} {e['level']:8s} {e['accession']:16s} 16S={e['length_bp']} bp")

V = np.array([table["entries"][k]["vector"] for k in keys])
assert np.isfinite(V).all() and V.shape[1] == table["embedding_dim"]
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
print("\ncosine similarity between organisms (species-aware embeddings should differ):")
print(np.round(Vn @ Vn.T, 3))

In [ ]:
# Save to Drive and download -- commit this file to the repo.
import shutil
from google.colab import drive, files

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/soamp_cache", exist_ok=True)
shutil.copy("data/organism_16s_dnabert_s.json", "/content/drive/MyDrive/soamp_cache/organism_16s_dnabert_s.json")
files.download("data/organism_16s_dnabert_s.json")